# Contested Norms Study
## Data Prep: Extract data from the CivilServant database

Created: 2025-09-04  
Updated: 2025-10-08
Authors: Edward L. Platt  

In [ ]:
%matplotlib inline
from configparser import ConfigParser
import csv
from datetime import datetime, timedelta
import logging
import math
import os
import pytz
import simplejson as json
import sys
from tqdm import tqdm
utc=pytz.UTC

### LOAD SQLALCHEMY
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy import text, and_, or_
import sqlalchemy.orm.session

### LOAD CONFIGURATION
config = ConfigParser()
config.read('config.ini')
config.write(sys.stdout)

subreddit_id = config.get("Source", "subreddit_id")
start_time = utc.localize(datetime.strptime(config.get("Source", "start_time"), "%Y-%m-%d %H:%M:%S"))
end_time = utc.localize(datetime.strptime(config.get("Source", "end_time"), "%Y-%m-%d %H:%M:%S"))

script_name = "contested_norms-{}-extract".format(subreddit_id)
script_date = datetime.now().strftime('%Y-%m-%d')

# Configure logging
logging.basicConfig(
    filename='{}-{}.log'.format(script_date, script_name),
    format='%(asctime)s:%(levelname)s:%(message)s',
    level=logging.DEBUG)
logger = logging.getLogger("CivilServant-Analysis")
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s:%(levelname)s:%(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

ENV = "jupyter"
os.environ['CS_ENV'] = ENV
BASE_DIR = "/cs/civilservant-jupyter"
sys.path.append(BASE_DIR)

### LOAD CIVILSERVANT
from app.models import Post, Comment, ModAction, Subreddit

### LOAD ANALYSIS CODE
from pipeline import DataFile
from pipeline.sources import (
    CivilServantPostSource,
    CivilServantCommentSource,
    CivilServantModActionSource,
    PRAWPostSource,
    PRAWCommentSource)

In [ ]:
with open(os.path.join(BASE_DIR, "config") + "/{env}.json".format(env=ENV), "r") as config:
  DBCONFIG = json.loads(config.read())

In [ ]:
source = CivilServantPostSource(subreddit_id, start_time, end_time, DBCONFIG)
posts = DataFile(source)
posts.extract()

In [ ]:
source = CivilServantCommentSource(subreddit_id, start_time, end_time, DBCONFIG)
comments = DataFile(source)
comments.extract()

In [ ]:
source = CivilServantModActionSource(
    subreddit_id, start_time, end_time, DBCONFIG)
modactions = DataFile(source)
modactions.extract()

In [ ]:
config.read('config.ini')
post_file = config.get("Extracted", "post_file")
comment_file = config.get("Extracted", "comment_file")
modaction_file = config.get("Extracted", "modaction_file")

post_source = CivilServantPostSource(subreddit_id, start_time, end_time)
posts = DataFile(
    post_source, filename=post_file, load=True)

comment_source = CivilServantCommentSource(subreddit_id, start_time, end_time)
comments = DataFile(
    comment_source, filename=comment_file, load=True)

In [ ]:
modaction_source = CivilServantModActionSource(subreddit_id, start_time, end_time)
modactions = DataFile(modaction_source, filename=modaction_file, load=True)

In [ ]:

civilservant_posts = set([post['id'] for post in posts.rows()])
civilservant_comments = set([comment['id'] for comment in comments.rows()])
modaction_rows = modactions.rows()

logger.info("Identifying missing posts/comments from moderation log")
missing_posts = set()
missing_comments = set()
for modaction in modaction_rows:
    if modaction['action'] in ['removelink', 'approvelink', 'spamlink']:
        if modaction['target.fullname'].replace("t3_", "") not in civilservant_posts:
            missing_posts.add(modaction['target.fullname'])
    elif modaction['action'] in ['removecomment', 'approvecomment', 'spamcomment']:
        if modaction['target.fullname'].replace("t1_", "") not in civilservant_comments:
            missing_comments.add(modaction['target.fullname'])
logger.info("  {} posts".format(len(missing_posts)))
logger.info("  {} comments".format(len(missing_comments)))

In [ ]:
### LOAD PRAW
import configparser
import time
import pickle
import praw
from praw.errors import NotFound
# Read praw.ini, for some reason praw can't find it
config = configparser.ConfigParser()
config.read('/cs/civilservant-jupyter/praw.ini')
# Read auth info from pickle
with open('/cs/civilservant-jupyter/config/access_information_jupyter.pickle', 'rb') as f:
    info = pickle.loads(f.read())
# Create PRAW object
r = praw.Reddit(
    user_agent=config['DEFAULT']['user_agent'],
    client_id=config['DEFAULT']['oauth_client_id'],
    client_secret=config['DEFAULT']['oauth_client_secret'],
    refresh_token=info['refresh_token'])

In [ ]:
source = PRAWPostSource(
    subreddit_id,
    start_time,
    end_time,
    r,
    missing_posts,
    delay_s=5)
praw_posts = DataFile(source)
praw_posts.extract()

In [ ]:
source = PRAWCommentSource(
    subreddit_id,
    start_time,
    end_time,
    r,
    missing_comments,
    delay_s=5)
praw_comments = DataFile(source)
praw_comments.extract()